In [1]:
rsnapsim_dir = r'C:\Users\Willi\Documents\Github\rSNAPsim'
rsnaped_dir = r'C:\Users\Willi\Documents\Github\rSNAPed'
gene_file = r'C:\Users\Willi\Downloads\pNZ266(pUB-24xGCN4-KDM5B-MS2).dna'

# Imports

In [2]:
%%capture
import os; from os import listdir; from os.path import isfile, join
import re  
from skimage.io import imread
from skimage.exposure import rescale_intensity
import numpy as np 
from tqdm.notebook import tqdm
from timeit import default_timer as timer
import scipy
import pandas as pd
import shutil
import pathlib
import sys
import seaborn as sns
from skimage.measure import approximate_polygon
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300

import os
homedir = os.getcwd()
os.chdir(homedir)
os.chdir(rsnapsim_dir)
import rsnapsim as rsnp
os.chdir(homedir)

import scipy.stats as stats
import matplotlib.pyplot as plt 

plt.style.use("dark_background")

# Defining directories
current_dir = pathlib.Path(rsnaped_dir).absolute()
sequences_dir = current_dir.joinpath('DataBases','gene_files')
video_dir = current_dir.joinpath('DataBases','videos_for_sim_cell')
masks_dir = current_dir.joinpath('DataBases','masks_for_sim_cell')

# Importing rSNAPed
sys.path.append(str(current_dir.joinpath('rsnaped')))
import rsnaped as rsp


from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.animation 
import numpy as np


import matplotlib.pyplot as plt
from cycler import cycler
import matplotlib.cm as cm
from matplotlib.lines import Line2D
colors = [ '#06d6a0','#ef476f', '#7400b8','#073b4c', '#118ab2',]
#colors = ['#fa8174', '#b3de69', '#bc82bd','#ccebc4','#ffed6f','#81b1d2']
font = {'family' : 'sans-serif',
        'weight' : 'bold',
        'size'   : 12}
plt.rcParams.update({'font.size': 12, 'font.weight':'bold', 'font.family':'sans-serif'  }   )
plt.rcParams.update({'axes.prop_cycle':cycler(color=colors)})
plt.rcParams.update({'axes.prop_cycle':cycler(color=colors)})
plt.rcParams.update({'axes.prop_cycle':cycler(color=colors)})
plt.rcParams.update({'xtick.major.width'   : 2.8 })
plt.rcParams.update({'xtick.labelsize'   : 12 })
plt.rcParams.update({'ytick.major.width'   : 2.8 })
plt.rcParams.update({'ytick.labelsize'   : 12})
plt.rcParams.update({'axes.titleweight'   : 'bold'})
plt.rcParams.update({'axes.titlesize'   : 10})
plt.rcParams.update({'axes.labelweight'   : 'bold'})
plt.rcParams.update({'axes.labelsize'   : 12})
plt.rcParams.update({'axes.linewidth':2.8})
plt.rcParams.update({'axes.labelpad':8})
plt.rcParams.update({'axes.titlepad':10})


In [3]:
#@title plasmid
plasmid = '''AGCUUGCCUUGAGUGCUCAAAGUAGUGUGUGCCCGUCUGUUGUGUGACUCUGGUAACUAGAGAUCCCUCAGACCCUUUUAGUCAGUGUGGAAAAUCUCUAGCAGUGGCGCCCGAACAGGGACUUGAAAGCGAAAGUAAAGCCAGAGGAGAUCUCUCGACGCAGGACUCGGCUUGCUGAAGCGCGCACGGCAAGAGGCGAGGGGCGGCGACUGGUGAGUACGCCAAAAAUUUUGACUAGCGGAGGCUAGAAGGAGAGAGAUGGGUGCGAGAGCGUCGGUAUUAAGCGGGGGAGAAUUAGAUAAAUGGGAAAAAAUUCGGUUAAGGCCAGGGGGAAAGAAACAAUAUAAACUAAAACAUAUAGUAUGGGCAAGCAGGGAGCUAGAACGAUUCGCAGUUAAUCCUGGCCUUUUAGAGACAUCAGAAGGCUGUAGACAAAUACUGGGACAGCUACAACCAUCCCUUCAGACAGGAUCAGAAGAACUUAGAUCAUUAUAUAAUACAAUAGCAGUCCUCUAUUGUGUGCAUCAAAGGAUAGAUGUAAAAGACACCAAGGAAGCCUUAGAUAAGAUAGAGGAAGAGCAAAACAAAAGUAAGAAAAAGGCACAGCAAGCAGCAGCUGACACAGGAAACAACAGCCAGGUCACCACGCGUUCUGGAGGAGAAGAACUUUUGAGCAAGAAUUAUCAUCUUGAGAACGAAGUGGCUCGUCUUAAGAAAGGUUCUGGCAGUGGAGAAGAACUGCUUUCAAAGAAUUACCACCUGGAAAAUGAGGUAGCUAGACUGAAAAAGGGGAGCGGAAGUGGGGAGGAGUUGCUGAGCAAAAAUUAUCAUUUGGAGAACGAAGUAGCACGACUAAAGAAAGGGUCCGGAUCGGGUGAGGAGUUACUCUCGAAAAAUUAUCAUCUCGAAAACGAAGUGGCUCGGCUAAAAAAGGGCAGUGGUUCUGGAGAAGAGCUAUUAUCUAAAAACUACCACCUCGAAAAUGAGGUGGCACGCUUAAAAAAGGGAAGUGGCAGUGGUGAAGAGCUACUAUCCAAGAAUUAUCAUCUUGAGAACGAGGUAGCGCGUUUGAAGAAGGGUUCCGGCUCAGGAGAGGAACUGCUCUCGAAGAACUAUCAUCUUGAAAAUGAGGUCGCUCGAUUAAAAAAGGGAUCGGGCAGUGGUGAGGAACUACUUUCAAAGAAUUACCACCUCGAAAACGAAGUAGCUCGAUUAAAGAAAGGUUCAGGGUCGGGUGAAGAAUUACUGAGUAAAAAUUAUCAUCUGGAAAAUGAGGUAGCGAGACUAAAAAAGGGGAGUGGUUCUGGCGAAGAGUUGCUAUCGAAAAAUUAUCAUCUUGAGAACGAAGUUGCUAGGCUCAAAAAGGGCUCAGGCUCAGGCGAGGAGUUGCUCUCGAAAAACUACCACUUGGAAAAUGAGGUCGCGAGGUUGAAAAAGGGGAGCGGGUCGGGCGAGGAGUUAUUGAGCAAAAACUAUCAUUUAGAGAACGAAGUCGCGCGCUUAAAGAAAGGCUCGGGCUCGGGCGAAGAACUCUUAUCGAAGAACUACCACCUCGAAAAUGAGGUCGCCAGGUUGAAAAAGGGCAGUGGCAGCGGGGAGGAACUCUUGAGCAAGAACUACCACUUGGAGAAUGAGGUCGCGAGAUUGAAGAAAGGGUCGGGGAGCGGCGAGGAAUUGCUCAGCAAGAAUUAUCAUUUGGAGAACGAAGUCGCCAGGCUCAAGAAAGGCUCGGGGUCGGGGGAGGAAUUGUUGAGUAAAAACUACCACUUGGAAAAUGAAGUCGCCAGGCUCAAAAAAGGGAGUGGGAGCGGCGAAGAGUUAUUGAGCAAAAAUUACCACUUGGAGAACGAAGUGGCAAGGCUCAAGAAAGGGAGCGGCAGCGGGGAGGAGCUCUUAUCGAAGAACUACCACUUAGAGAAUGAAGUCGCCCGCUUGAAGAAAGGCUCGGGGAGCGGGGAAGAGCUCUUGAGCAAGAACUACCACUUGGAAAAUGAGGUGGCGCGCUUGAAGAAAGGGAGCGGGAGCGGGGAAGAGUUACUAUCUAAGAAUUAUCAUCUCGAGAACGAGGUGGCUCGACUAAAGAAGGGCUCCGGCAGUGGGGAGGAACUCCUGUCGAAGAACUAUCAUCUUGAAAAUGAGGUUGCAAGACUUAAAAAGGGGUCCGGAUCAGGUGAGGAACUACUCAGUGCUAGCUCGCGUGUCAGCCAAAAUUACCCUAUAGUGCAGAACCUCCAGGGGCAAAUGGUACAUCAGGCCAUAUCACCUAGAACUUUAAAUGCAUGGGUAAAAGUAGUAGAAGAGAAGGCUUUCAGCCCAGAAGUAAUACCCAUGUUUUCAGCAUUAUCAGAAGGAGCCACCCCACAAGAUUUAAAUACCAUGCUAAACACAGUGGGGGGACAUCAAGCAGCCAUGCAAAUGUUAAAAGAGACCAUCAAUGAGGAAGCUGCAGAAUGGGAUAGAUUGCAUCCAGUGCAUGCAGGGCCUAUUGCACCAGGCCAGAUGAGAGAACCAAGGGGAAGUGACAUAGCAGGAACUACUAGUACCCUUCAGGAACAAAUAGGAUGGAUGACACAUAAUCCACCUAUCCCAGUAGGAGAAAUCUAUAAAAGAUGGAUAAUCCUGGGAUUAAAUAAAAUAGUAAGAAUGUAUAGCCCUACCAGCAUUCUGGACAUAAGACAAGGACCAAAGGAACCCUUUAGAGACUAUGUAGACCGAUUCUAUAAAACUCUAAGAGCCGAGCAAGCUUCACAAGAGGUAAAAAAUUGGAUGACAGAAACCUUGUUGGUCCAAAAUGCGAACCCAGAUUGUAAGACUAUUUUAAAAGCAUUGGGACCAGGAGCGACACUAGAAGAAAUGAUGACAGCAUGUCAGGGAGUGGGGGGACCCGGCCAUAAAGCAAGAGUUUUGGCUGAAGCAAUGAGCCAAGUAACAAAUCCAGCUACCAUAAUGAUACAGAAAGGCAAUUUUAGGAACCAAAGAAAGACUGUUAAGUGUUUCAAUUGUGGCAAAGAAGGGCACAUAGCCAAAAAUUGCAGGGCCCCUAGGAAAAAGGGCUGUUGGAAAUGUGGAAAGGAAGGACACCAAAUGAAAGAUUGUACUGAGAGACAGGCUAAUUUUUUAGGGAAGAUCUGGCCUUCCCACAAGGGAAGGCCAGGGAAUUUUCUUCAGAGCAGACCAGAGCCAACAGCCCCACCAGAAGAGAGCUUCAGGUUUGGGGAAGAGACAACAACUCCCUCUCAGAAGCAGGAGCCGAUAGACAAGGAACUGUAUCCUUUAGCUUCCCUCAGAUCACUCUUUGGCAGCGACCCCUCGUCACAAUAAAGAUAGGGGGGCAAUUAAAGGAAGCUCUAUUAGAUACAGGAGCAGAUGAUACAGUAUUAGAAGAAAUGAAUUUGCCAGGAAGAUGGAAACCAAAAAUGAUAGGGGGAAUUGGAGGUUUUAUCAAAGUAAGACAGUAUGAUCAGAUACUCAUAGAAAUCUGCGGACAUAAAGCUAUAGGUACAGUAUUAGUAGGACCUACACCUGUCAACAUAAUUGGAAGAAAUCUGUUGACUCAGAUUGGCUGCACUUUAAAUUUUCCCAUUAGUCCGCGGAUGGAUUACAAGGAUGACGACGAUAAGGGUUCUGGCAGUGGAGAUUACAAAGACGAUGAUGACAAGGGGAGCGGAAGUGGGGACUACAAGGACGACGACGACAAGGGGUCCGGAUCGGGUGACUACAAAGAUGACGAUGAUAAAGGAGGCGGUCAGCUGGACUACAAGGAUCACGACGGAGACUACAAGGACCACGAUAUCGAUUACAAGGAUGACGAUGAUAAGCUUAUUGAGACUGUACCAGUAAAAUUAAAGCCAGGAAUGGAUGGCCCAAAAGUUAAACAAUGGCCAUUGACAGAAGAAAAAAUAAAAGCAUUAGUAGAAAUUUGUACAGAAAUGGAAAAGGAAGGAAAAAUUUCAAAAAUUGGGCCUGAAAAUCCAUACAAUACUCCAGUAUUUGCCAUAAAGAAAAAAGACAGUACUAAAUGGAGAAAAUUAGUAGAUUUCAGAGAACUUAAUAAGAGAACUCAAGAUUUCUGGGAAGUUCAAUUAGGAAUACCACAUCCUGCAGGGUUAAAACAGAAAAAAUCAGUAACAGUACUGGAUGUGGGCGAUGCAUAUUUUUCAGUUCCCUUAGAUAAAGACUUCAGGAAGUAUACUGCAUUUACCAUACCUAGUAUAAACAAUGAGACACCAGGGAUUAGAUAUCAGUACAAUGUGCUUCCACAGGGAUGGAAAGGAUCACCAGCAAUAUUCCAGUGUAGCAUGACAAAAAUCUUAGAGCCUUUUAGAAAACAAAAUCCAGACAUAGUCAUCUAUCAAUACAUGGAUGAUUUGUAUGUAGGAUCUGACUUAGAAAUAGGGCAGCAUAGAACAAAAAUAGAGGAACUGAGACAACAUCUGUUGAGGUGGGGAUUUACCACACCAGACAAAAAACAUCAGAAAGAACCUCCAUUCCUUUGGAUGGGUUAUGAACUCCAUCCUGAUAAAUGGACAGUACAGCCUAUAGUGCUGCCAGAAAAGGACAGCUGGACUGUCAAUGACAUACAGAAAUUAGUGGGAAAAUUGAAUUGGGCAAGUCAGAUUUAUGCAGGGAUUAAAGUAAGGCAAUUAUGUAAACUUCUUAGGGGAACCAAAGCACUAACAGAAGUAGUACCACUAACAGAAGAAGCAGAGCUAGAACUGGCAGAAAACAGGGAGAUUCUAAAAGAACCGGUACAUGGAGUGUAUUAUGACCCAUCAAAAGACUUAAUAGCAGAAAUACAGAAGCAGGGGCAAGGCCAAUGGACAUAUCAAAUUUAUCAAGAGCCAUUUAAAAAUCUGAAAACAGGAAAAUAUGCAAGAAUGAAGGGUGCCCACACUAAUGAUGUGAAACAAUUAACAGAGGCAGUACAAAAAAUAGCCACAGAAAGCAUAGUAAUAUGGGGAAAGACUCCUAAAUUUAAAUUACCCAUACAAAAGGAAACAUGGGAAGCAUGGUGGACAGAGUAUUGGCAAGCCACCUGGAUUCCUGAGUGGGAGUUUGUCAAUACCCCUCCCUUAGUGAAGUUAUGGUACCAGUUAGAGAAAGAACCCAUAAUAGGAGCAGAAACUUUCUAUGUAGAUGGGGCAGCCAAUAGGGAAACUAAAUUAGGAAAAGCAGGAUAUGUAACUGACAGAGGAAGACAAAAAGUUGUCCCCCUAACGGACACAACAAAUCAGAAGACUGAGUUACAAGCAAUUCAUCUAGCUUUGCAGGAUUCGGGAUUAGAAGUAAACAUAGUGACAGACUCACAAUAUGCAUUGGGAAUCAUUCAAGCACAACCAGAUAAGAGUGAAUCAGAGUUAGUCAGUCAAAUAAUAGAGCAGUUAAUAAAAAAGGAAAAAGUCUACCUGGCAUGGGUACCAGCACACAAAGGAAUUGGAGGAAAUGAACAAGUAGAUGGGUUGGUCAGUGCUGGAAUCAGGAAAGUACUAUUUUUAGAUGGAAUAGAUAAGGCCCAAGAAGAACAUGAGAAAUAUCACAGUAAUUGGAGAGCAAUGGCUAGUGAUUUUAACCUACCACCUGUAGUAGCAAAAGAAAUAGUAGCCAGCUGUGAUAAAUGUCAGCUAAAAGGGGAAGCCAUGCAUGGACAAGUAGACUGUAGCCCAGGAAUAUGGCAGCUAGAUUGUACACAUUUAGAAGGAAAAGUUAUCUUGGUAGCAGUUCAUGUAGCCAGUGGAUAUAUAGAAGCAGAAGUAAUUCCAGCAGAGACAGGGCAAGAAACAGCAUACUUCCUCUUAAAAUUAGCAGGAAGAUGGCCAGUAAAAACAGUACAUACAGACAAUGGCAGCAAUUUCACCAGUACUACAGUUAAGGCCGCCUGUUGGUGGGCGGGGAUCAAGCAGGAAUUUGGCAUUCCCUACAAUCCCCAAAGUCAAGGAGUAAUAGAAUCUAUGAAUAAAGAAUUAAAGAAAAUUAUAGGACAGGUAAGAGAUCAGGCUGAACAUCUUAAGACAGCAGUACAAAUGGCAGUAUUCAUCCACAAUUUUAAAAGAAAAGGGGGGAUUGGGGGGUACAGUGCAGGGGAAAGAAUAGUAGACAUAAUAGCAACAGACAUACAAACUAAAGAAUUACAAAAACAAAUUACAAAAAUUCAAAAUUUUCGGGUUUAUUACAGGGACAGCAGAGAUCCAGUUUGGAAAGGACCAGCAAAGCUCCUCUGGAAAGGUGAAGGGGCAGUAGUAAUACAAGAUAAUAGUGACAUAAAAGUAGUGCCAAGAAGAAAAGCAAAGAUCAUCAGGGAUUAUGGAAAACAGAUGGCAGGUGAUGAUUGUGUGGCAAGUAGACAGGAUGAGGAUUAACACAUGGAAAAGAUUAGUAAAACACCAUAUGUAUAUUUCAAGGAAAGCUAAGGACUGGUUUUAUAGACAUCACUAUGAAAGUACUAAUCCAAAAAUAAGUUCAGAAGUACACAUCCCACUAGGGGAUGCUAAAUUAGUAAUAACAACAUAUUGGGGUCUGCAUACAGGAGAAAGAGACUGGCAUUUGGGUCAGGGAGUCUCCAUAGAAUGGAGGAAAAAGAGAUAUAGCACACAAGUAGACCCUGACCUAGCAGACCAACUAAUUCAUCUGCACUAUUUUGAUUGUUUUUCAGAAUCUGCUAUAAGAAAUACCAUAUUAGGACGUAUAGUUAGUCCUAGGUGUGAAUAUCAAGCAGGACAUAACAAGGUAGGAUCUCUACAGUACUUGGCACUAGCAGCAUUAAUAAAACCAAAACAGAUAAAGCCACCUUUGCCUAGUGUUAGGAAACUGACAGAGGACAGAUGGAACAAGCCCCAGAAGACCAAGGGCCACAGAGGGAGCCAUACAAUGAAUGGACACUAGAGCUUUUAGAGGAACUUAAGAGUGAAGCUGUUAGACAUUUUCCUAGGAUAUGGCUCCAUAACUUAGGACAACAUAUCUAUGAAACUUACGGGGAUACUUGGGCAGGAGUGGAAGCCAUAAUAAGAAUUCUGCAACAACUGCUGUUUAUCCAUUUCAGAAUUGGGUGUCGACAUAGCAGAAUAGGCGUUACUCGACAGAGGAGAGCAAGAAAUGGAGCCAGUAGAUCCUAGACUAGAGCCCUGGAAGCAUCCAGGAAGUCAGCCUAAAACUGCUUGUACCAAUUGCUAUUGUAAAAAGUGUUGCUUUCAUUGCCAAGUUUGUUUCAUGACAAAAGCCUUAGGCAUCUCCUAUGGCAGGAAGAAGCGGAGACAGCGACGAAGAGCUCAUCAGAACAGUCAGACUCAUCAAGCUUCUCUAUCAAAGCAGUAAGUAGUACAUGGGCGCGCCCAUGUGGCAGGAAGUAGGAAAAGCAAUGUAUGCCCCUCCCAUCAGUGGACAAAUUAGAUGUUCAUCAAAUAUUACUGGGCUGCUAUUAACAAGAGAUGGUGGUAAUAACAACAAUGGGUCCGAGAUCUUCAGACCUGGAGGAGGCGAUAUGAGGGACAAUUGGAGAAGUGAAUUAUAUAAAUAUAAAGUAGUAAAAAUUGAACCAUUAGGAGUAGCACCCACCAAGGCAAAGAGAAGAGUGGUGCAGAGAGAAAAAAGAGCAGUGGGAAUAGGAGCUUUGUUCCUUGGGUUCUUGGGAGCAGCAGGAAGCACUAUGGGCGCAGCGUCAAUGACGCUGACGGUACAGGCCAGACAAUUAUUGUCUGAUAUAGUGCAGCAGCAGAACAAUUUGCUGAGGGCUAUUGAGGCGCAACAGCAUCUGUUGCAACUCACAGUCUGGGGCAUCAAACAGCUCCAGGCAAGAAUCCUGGCUGUGGAAAGAUACCUAAAGGAUCAACAGCUCCUGGGGAUUUGGGGUUGCUCUGGAAAACUCAUUUGCACCACUGCUGUGCCUUGGAAUGCUAGUUGGAGUAAUAAAUCUCUGGAACAGAUUUGGAAUAACAUGACCUGGAUGGAGUGGGACAGAGAAAUUAACAAUUACACAAGCUUAAUACACUCCUUAAUUGAAGAAUCGCAAAACCAGCAAGAAAAGAAUGAACAAGAAUUAUUGGAAUUAGAUAAAUGGGCAAGUUUGUGGAAUUGGUUUAACAUAACAAAUUGGCUGUGGUAUAUAAAAUUAUUCAUAAUGAUAGUAGGAGGCUUGGUAGGUUUAAGAAUAGUUUUUGCUGUACUUUCUAUAGUGAAUAGAGUUAGGCAGGGAUAUUCACCAUUAUCGUUUCAGACCCACCUCCCAAUCCCGAGGGGACCCGACAGGCCCGAAGGAAUAGAAGAAGAAGGUGGAGAGAGAGACAGAGACAGAUCCAUUCGAUUAGUGAACGGAUCCUUAGCACUUAUCUGGGACGAUCUGCGGAGCCUGUGCCUCUUCAGCUACCACCGCUUGAGAGACUUACUCUUGAUUGUAACGAGGAUUGUGGAACUUCUGGGACGCAGGGGGUGGGAAGCCCUCAAAUAUUGGUGGAAUCUCCUACAGUAUUGGAGUCAGGAACUAAAGAAUAGUGCUGUUAACUUGCUCAAUGCCACAGCCAUAGCAGUAGCUGAGGGGACAGAUAGGGUUAUAGAAGUAUUACAAGCAGCUUAUAGAGCUAUUCGCCACAUACCUAGAAGAAUAAGACAGGGCUUGGAAAGGAUUUUGCUAUAAGAUGGGUGGCGCGGCCGCUCCAGAGCCAGCGAAGUCUGCUCCCGCCCCGAAAAAGGGCUCCAAGAAGGCGGUGACUAAGGCGCAGAAGAAAGGCGGCAAGAAGCGCAAGCGCAGCCGCAAGGAGAGCUAUUCCAUCUAUGUGUACAAGGUUCUGAAGCAGGUCCACCCUGACACCGGCAUUUCGUCCAAGGCCAUGGGCAUCAUGAAUUCGUUUGUGAACGACAUUUUCGAGCGCAUCGCAGGUGAGGCUUCCCGCCUGGCGCAUUACAACAAGCGCUCGACCAUCACCUCCAGGGAGAUCCAGACGGCCGUGCGCCUGCUGCUGCCUGGGGAGUUGGCCAAGCACGCCGUGUCCGAGGGUACUAAGGCCAUCACCAAGUACACCAGCGCUAAGGGCUCAGCCUCCUCCGAGGACGUCAUCAAGGAGUUCAUGCGCUUCAAGGUGCGCAUGGAGGGCUCCGUGAACGGCCACGAGUUCGAGAUCGAGGGCGAGGGCGAGGGCCGCCCCUACGAGGGCACCCAGACCGCCAAGCUGAAGGUGACCAAGGGCGGCCCCCUGCCCUUCGCCUGGGACAUCCUGUCCCCUCAGUUCCAGUACGGCUCCAAGGCCUACGUGAAGCACCCCGCCGACAUCCCCGACUACUUGAAGCUGUCCUUCCCCGAGGGCUUCAAGUGGGAGCGCGUGAUGAACUUCGAGGACGGCGGCGUGGUGACCGUGACCCAGGACUCCUCCCUGCAGGACGGCGAGUUCAUCUACAAGGUGAAGCUGCGCGGCACCAACUUCCCCUCCGACGGCCCCGUAAUGCAGAAGAAGACCAUGGGCUGGGAGGCCUCCACCGAGCGGAUGUACCCCGAGGACGGCGCCCUGAAGGGCGAGAUCAAGAUGAGGCUGAAGCUGAAGGACGGCGGCCACUACGACGCCGAGGUCAAGACCACCUACAUGGCCAAGAAGCCCGUGCAGCUGCCCGGCGCCUACAAGACCGACAUCAAGCUGGACAUCACCUCCCACAACGAGGACUACACCAUCGUGGAACAGUACGAGCGCGCCGAGGGCCGCCACUCCACCGGCGCCUAGCUCGAGACCUAGAAAAACAUGGAGCAAUCACAAGUAGCAAUACAGCAGCUAACAAUGCUGCUUGUGCCUGGCUAGAAGCACAAGAGGAGGAAGAGGUGGGUUUUCCAGUCACACCUCAGGUACCUUUAAGACCAAUGACUUACAAGGCAGCUGUAGAUCUUAGCCACUUUUUAAAAGAAAAGGGGGGACUGGAAGGGCUAAUUCACUCCCAAAGAAGACAAGAUAUCCUUGAUCUGUGGAUCUACCACACACAAGGCUACUUCCCUGAUUGGCAGAACUACACACCAGGGCCAGGGGUCAGAUAUCCACUGACCUUUGGAUGGUGCUACAAGCUAGUACCAGUUGAGCCAGAUAAGGUAGAAGAGGCCAAUAAAGGAGAGAACACCAGCUUGUUACACCCUGUGAGCCUGCAUGGAAUGGAUGACCCUGAGAGAGAAGUGUUAGAGUGGAGGUUUGACAGCCGCCUAGCAUUUCAUCACGUGGCCCGAGAGCUGCAUCCGGAGUACUUCAAGAACUGCUGACAUCGAGCUUGCUACAAGGGACUUUCCGCUGGGGACUUUCCAGGGAGGCGUGGCCUGGGCGGGACUGGGGAGUGGCGAGCCCUCAGAUGCUGCAUAUAAGCAGCUGCUUUUUGCCUGUACUGGGUCUCUCUGGUUAGACCAGAUCUGAGCCUGGGAGCUCUCUGGCUAACUAGGGAACCCACUGCUUAAGCCUCAAUAAAGCUUGCCUUGAGUGCUUCAAGUAGUGUGUGCCCGUCUGUUGUGUGACUCUGGUAACUAGAGAUCCCUCAGACCCUUUUAGUCAGUGUGGAAAAUCUCUAGCACCCCCCAGGAGGUAGAGGUUGCAGUGAGCCAAGAUCGCGCCACUGCAUUCCAGCCUGGGCAAGAAAACAAGACUGUCUAAAAUAAUAAUAAUAAGUUAAGGGUAUUAAAUAUAUUUAUACAUGGAGGUCAUAAAAAUAUAUAUAUUUGGGCUGGGCGCAGUGGCUCACACCUGCGCCCGGCCCUUUGGGAGGCCGAGGCAGGUGGAUCACCUGAGUUUGGGAGUUCCAGACCAGCCUGACCAACAUGGAGAAACCCCUUCUCUGUGUAUUUUUAGUAGAUUUUAUUUUAUGUGUAUUUUAUUCACAGGUAUUUCUGGAAAACUGAAACUGUUUUUCCUCUACUCUGAUACCACAAGAAUCAUCAGCACAGAGGAAGACUUCUGUGAUCAAAUGUGGUGGGAGAGGGAGGUUUUCACCAGCACAUGAGCAGUCAGUUCUGCCGCAGACUCGGCGGGUGUCCUUCGGUUCAGUUCCAACACCGCCUGCCUGGAGAGAGGUCAGACCACAGGGUGAGGGCUCAGUCCCCAAGACAUAAACACCCAAGACAUAAACACCCAACAGGUCCACCCCGCCUGCUGCCCAGGCAGAGCCGAUUCACCAAGACGGGAAUUAGGAUAGAGAAAGAGUAAGUCACACAGAGCCGGCUGUGCGGGAGAACGGAGUUCUAUUAUGACUCAAAUCAGUCUCCCCAAGCAUUCGGGGAUCAGAGUUUUUAAGGAUAACUUAGUGUGUAGGGGGCCAGUGAGUUGGAGAUGAAAGCGUAGGGAGUCGAAGGUGUCCUUUUGCGCCGAGUCAGUUCCUGGGUGGGGGCCACAAGAUCGGAUGAGCCAGUUUAUCAAUCCGGGGGUGCCAGCUGAUCCAUGGAGUGCAGGGUCUGCAAAAUAUCUCAAGCACUGAUUGAUCUUAGGUUUUACAAUAGUGAUGUUACCCCAGGAACAAUUUGGGGAAGGUCAGAAUCUUGUAGCCUGUAGCUGCAUGACUCCUAAACCAUAAUUUCUUUUUUGUUUUUUUUUUUUUAUUUUUGAGACAGGGUCUCACUCUGUCACCUAGGCUGGAGUGCAGUGGUGCAAUCACAGCUCACUGCAGCCUCAACGUCGUAAGCUCAAGCGAUCCUCCCACCUCAGCCUGCCUGGUAGCUGAGACUACAAGCGACGCCCCAGUUAAUUUUUGUAUUUUUGGUAGAGGCAGCGUUUUGCCGUGUGGCCCUGGCUGGUCUCGAACUCCUGGGCUCAAGUGAUCCAGCCUCAGCCUCCCAAAGUGCUGGGACAACCGGGGCCAGUCACUGCACCUGGCCCUAAACCAUAAUUUCUAAUCUUUUGGCUAAUUUGUUAGUCCUACAAAGGCAGUCUAGUCCCCAGGCAAAAAGGGGGUUUGUUUCGGGAAAGGGCUGUUACUGUCUUUGUUUCAAACUAUAAACUAAGUUCCUCCUAAACUUAGUUCGGCCUACACCCAGGAAUGAACAAGGAGAGCUUGGAGGUUAGAAGCACGAUGGAAUUGGUUAGGUCAGAUCUCUUUCACUGUCUGAGUUAUAAUUUUGCAAUGGUGGUUCAAAGACUGCCCGCUUCUGACACCAGUCGCUGCAUUAAUGAAUCGGCCAACGCGCGGGGAGAGGCGGUUUGCGUAUUGGCGCUCUUCCGCUUCCUCGCUCACUGACUCGCUGCGCUCGGUCGUUCGGCUGCGGCGAGCGGUAUCAGCUCACUCAAAGGCGGUAAUACGGUUAUCCACAGAAUCAGGGGAUAACGCAGGAAAGAACAUGUGAGCAAAAGGCCAGCAAAAGGCCAGGAACCGUAAAAAGGCCGCGUUGCUGGCGUUUUUCCAUAGGCUCCGCCCCCCUGACGAGCAUCACAAAAAUCGACGCUCAAGUCAGAGGUGGCGAAACCCGACAGGACUAUAAAGAUACCAGGCGUUUCCCCCUGGAAGCUCCCUCGUGCGCUCUCCUGUUCCGACCCUGCCGCUUACCGGAUACCUGUCCGCCUUUCUCCCUUCGGGAAGCGUGGCGCUUUCUCAAUGCUCACGCUGUAGGUAUCUCAGUUCGGUGUAGGUCGUUCGCUCCAAGCUGGGCUGUGUGCACGAACCCCCCGUUCAGCCCGACCGCUGCGCCUUAUCCGGUAACUAUCGUCUUGAGUCCAACCCGGUAAGACACGACUUAUCGCCACUGGCAGCAGCCACUGGUAACAGGAUUAGCAGAGCGAGGUAUGUAGGCGGUGCUACAGAGUUCUUGAAGUGGUGGCCUAACUACGGCUACACUAGAAGGACAGUAUUUGGUAUCUGCGCUCUGCUGAAGCCAGUUACCUUCGGAAAAAGAGUUGGUAGCUCUUGAUCCGGCAAACAAACCACCGCUGGUAGCGGUGGUUUUUUUGUUUGCAAGCAGCAGAUUACGCGCAGAAAAAAAGGAUCUCAAGAAGAUCCUUUGAUCUUUUCUACGGGGUCUGACGCUCAGUGGAACGAAAACUCACGUUAAGGGAUUUUGGUCAUGAGAUUAUCAAAAAGGAUCUUCACCUAGAUCCUUUUAAAUUAAAAAUGAAGUUUUAAAUCAAUCUAAAGUAUAUAUGAGUAAACUUGGUCUGACAGUUACCAAUGCUUAAUCAGUGAGGCACCUAUCUCAGCGAUCUGUCUAUUUCGUUCAUCCAUAGUUGCCUGACUCCCCGUCGUGUAGAUAACUACGAUACGGGAGGGCUUACCAUCUGGCCCCAGUGCUGCAAUGAUACCGCGAGACCCACGCUCACCGGCUCCAGAUUUAUCAGCAAUAAACCAGCCAGCCGGAAGGGCCGAGCGCAGAAGUGGUCCUGCAACUUUAUCCGCCUCCAUCCAGUCUAUUAAUUGUUGCCGGGAAGCUAGAGUAAGUAGUUCGCCAGUUAAUAGUUUGCGCAACGUUGUUGCCAUUGCUACAGGCAUCGUGGUGUCACGCUCGUCGUUUGGUAUGGCUUCAUUCAGCUCCGGUUCCCAACGAUCAAGGCGAGUUACAUGAUCCCCCAUGUUGUGCAAAAAAGCGGUUAGCUCCUUCGGUCCUCCGAUCGUUGUCAGAAGUAAGUUGGCCGCAGUGUUAUCACUCAUGGUUAUGGCAGCACUGCAUAAUUCUCUUACUGUCAUGCCAUCCGUAAGAUGCUUUUCUGUGACUGGUGAGUACUCAACCAAGUCAUUCUGAGAAUAGUGUAUGCGGCGACCGAGUUGCUCUUGCCCGGCGUCAAUACGGGAUAAUACCGCGCCACAUAGCAGAACUUUAAAAGUGCUCAUCAUUGGAAAACGUUCUUCGGGGCGAAAACUCUCAAGGAUCUUACCGCUGUUGAGAUCCAGUUCGAUGUAACCCACUCGUGCACCCAACUGAUCUUCAGCAUCUUUUACUUUCACCAGCGUUUCUGGGUGAGCAAAAACAGGAAGGCAAAAUGCCGCAAAAAAGGGAAUAAGGGCGACACGGAAAUGUUGAAUACUCAUACUCUUCCUUUUUCAAUAUUAUUGAAGCAUUUAUCAGGGUUAUUGUCUCAUGAGCGGAUACAUAUUUGAAUGUAUUUAGAAAAAUAAACAAAUAGGGGUUCCGCGCACAUUUCCCCGAAAAGUGCCACCUGACGUCUAAGAAACCAUUAUUAUCAUGACAUUAACCUAUAAAAAUAGGCGUAUCACGAGGCCCUUUCGUCUUCAAGAACUGCCUCGCGCGUUUCGGUGAUGACGGUGAAAACCUCUGACACAUGCAGCUCCCGGAGACGGUCACAGCUUGUCUGUAAGCGGAUGCCGGGAGCAGACAAGCCCGUCAGGGCGCGUCAGCGGGUGUUGGCGGGUGUCGGGGCGCAGCCAUGACCCAGUCACGUAGCGAUAGCGGAGUGUACUGGCUUAACUAUGCGGCAUCAGAGCAGAUUGUACUGAGAGUGCACCAUAUGCGGUGUGAAAUACCGCACAGAUGCGUAAGGAGAAAAUACCGCAUCAGGCGCCAUUCGCCAUUCAGGCUGCGCAACUGUUGGGAAGGGCGAUCGGUGCGGGCCUCUUCGCUAUUACGCCAGGGGAGGCAGAGAUUGCAGUAAGCUGAGAUCGCAGCACUGCACUCCAGCCUGGGCGACAGAGUAAGACUCUGUCUCAAAAAUAAAAUAAAUAAAUCAAUCAGAUAUUCCAAUCUUUUCCUUUAUUUAUUUAUUUAUUUUCUAUUUUGGAAACACAGUCCUUCCUUAUUCCAGAAUUACACAUAUAUUCUAUUUUUCUUUAUAUGCUCCAGUUUUUUUUAGACCUUCACCUGAAAUGUGUGUAUACAAAAUCUAGGCCAGUCCAGCAGAGCCUAAAGGUAAAAAAUAAAAUAAUAAAAAAUAAAUAAAAUCUAGCUCACUCCUUCACAUCAAAAUGGAGAUACAGCUGUUAGCAUUAAAUACCAAAUAACCCAUCUUGUCCUCAAUAAUUUUAAGCGCCUCUCUCCACCACAUCUAACUCCUGUCAAAGGCAUGUGCCCCUUCCGGGCGCUCUGCUGUGCUGCCAACCAACUGGCAUGUGGACUCUGCAGGGUCCCUAACUGCCAAGCCCCACAGUGUGCCCUGAGGCUGCCCCUUCCUUCUAGCGGCUGCCCCCACUCGGCUUUGCUUUCCCUAGUUUCAGUUACUUGCGUUCAGCCAAGGUCUGAAACUAGGUGCGCACAGAGCGGUAAGACUGCGAGAGAAAGAGACCAGCUUUACAGGGGGUUUAUCACAGUGCACCCUGACAGUCGUCAGCCUCACAGGGGGUUUAUCACAUUGCACCCUGACAGUCGUCAGCCUCACAGGGGGUUUAUCACAGUGCACCCUUACAAUCAUUCCAUUUGAUUCACAAUUUUUUUAGUCUCUACUGUGCCUAACUUGUAAGUUAAAUUUGAUCAGAGGUGUGUUCCCAGAGGGGAAAACAGUAUAUACAGGGUUCAGUACUAUCGCAUUUCAGGCCUCCACCUGGGUCUUGGAAUGUGUCCCCCGAGGGGUGAUGACUACCUCAGUUGGAUCUCCACAGGUCACAGUGACACAAGAUAACCAAGACACCUCCCAAGGCUACCACAAUGGGCCGCCCUCCACGUGCACAUGGCCGGAGGAACUGCCAUGUCGGAGGUGCAAGCACACCUGCGCAUCAGAGUCCUUGGUGUGGAGGGAGGGACCAGCGCAGCUUCCAGCCAUCCACCUGAUGAACAGAACCUAGGGAAAGCCCCAGUUCUACUUACACCAGGAAAGGCUGGAAGGGCUAAUUUGGUCCCAAAAAAGACAAGAGAUCCUUGAUCUGUGGAUCUACCACACACAAGGCUACUUCCCUGAUUGGCAGAACUACACACCAGGGCCAGGGAUCAGAUAUCCACUGACCUUUGGAUGGUGCUUCAAGUUAGUACCAGUUGAACCAGAGCAAGUAGAAGAGGCCAAAUAAGGAGAGAAGAACAGCUUGUUACACCCUAUGAGCCAGCAUGGGAUGGAGGACCCGGAGGGAGAAGUAUUAGUGUGGAAGUUUGACAGCCUCCUAGCAUUUCGUCACAUGGCCCGAGAGCUGCAUCCGGAGUACUACAAAGACUGCUGACAUCGAGCUUUCUACAAGGGACUUUCCGCUGGGGACUUUCCAGGGAGGUGUGGCCUGGGCGGGACUGGGGAGUGGCGAGCCCUCAGAUGCUACAUAUAAGCAGCUGCUUUUUGCCUGUACUGGGUCUCUCUGGUUAGACCAGAUCUGAGCCUGGGAGCUCUCUGGCUAACUAGGGAACCCACUGCUUAAGCCUCAAUAA'''.split('\n')

In [4]:
pois = rsnp.seqmanip.seq_to_CDS_obj(''.join(plasmid), add_tag=False)
GAG = pois['0'][0]
fminus1_start = 3111
POL = ''.join(plasmid)[fminus1_start+2:fminus1_start+2+1080*3+3]

POL = rsnp.seqmanip.seq_to_CDS_obj('AUG' + POL, add_tag=False)['0'][0]

GAG.tag_epitopes['T_Flag'] = [1] #hack to force 2 colors
GAG.generate_3frame_tags()                     # call to generate all open reading frames tags
GAG.multiframe_epitopes = [{'T_SunTag':[1],'T_Flag':[1]}, {},{}]

### Model 1: Gag-Pol w/ two state pausing 

This model frameshifts based on a two state bursting model, if state = 0 there is no FSS, if the state = 1, there is frameshifting in all ribosomes when they reach the HIV-FSS

In [15]:
ki = .0244
kon = 9.6e-5
koff = 1.3e-4
kpause_fss_off = .0234
#kelong[FSS_LOC] = 1/((1/kelong[FSS_LOC]) + (1/kpause_fss_off))
kpause_fss_on = .0139

kout_frame0 = 3
kout_frame1 = 3


aa_seq1 = GAG.aa_seq
aa_seq2 = POL.aa_seq[1:]
FSS_LOC = 951 #bp 3117 on plasmid start of UUUUUUU
lattice_length = len(aa_seq1) + len(aa_seq2) - (len(aa_seq1) - FSS_LOC)

parameters = [ki, kon, koff, kpause_fss_off, kpause_fss_on, kout_frame0, kout_frame1, len(aa_seq1), lattice_length, FSS_LOC, 1e6]


In [ ]:

aa_seq1 = GAG.aa_seq
aa_seq2 = POL.aa_seq[1:]


FSS_LOC = 951 #bp 3117 on plasmid start of UUUUUUU
lattice_length = len(aa_seq1) + len(aa_seq2) - (len(aa_seq1) - FSS_LOC)


#poi = seqmanip.seq_to_protein_obj(ken_sequence_frame_0) # convert a given sequence to a protein of interest object
mRNA = GAG                         # pull out the main open reading frame to make a blank model
mRNA_length = len(GAG.kelong)                  # get the length of the mRNA
model_pause_twostate = rsnp.tasep_model(mRNA,'GagPol_twostate_bursting_FSS') # model object

#manually editing the model
kelong_mat = np.zeros([3, lattice_length+1])
kelong_mat[0, :len(aa_seq1)] = rsnp.propf.get_k(GAG.nt_seq, ki, 3, 3)[1:-1]
kelong_mat[1, FSS_LOC:-1] = rsnp.propf.get_k(POL.nt_seq[3:], .1, 3, 3)[1:-1]

kelong_mat[0, len(aa_seq1)] = 0 #dont go past 0frame stop codon

kelong_mat[1,FSS_LOC] = kpause_fss_on # PAUSING DUE TO FSS SEQUENCE IN BOTH FRAMES
kelong_mat[0,FSS_LOC] = kpause_fss_off
kelong_mat[1,-2] = 3 # overwrite stop codon
kelong_mat[0,FSS_LOC+69] = 3 # overwrite stop codon

flagtags = np.array(POL.tag_epitopes['T_Flag'])-1 + FSS_LOC
suntags = GAG.tag_epitopes['T_SunTag']

probe_locations = np.zeros([3,lattice_length+1], dtype=np.intc  )
probe_locations[0,suntags] = 1
probe_locations[1,flagtags ] = 2
model_pause_twostate._probe_mat = probe_locations


#override the current kelongation mat
model_pause_twostate._kelong_mat = kelong_mat
model_pause_twostate._lattice_arr0 = np.zeros([lattice_length+1])
model_pause_twostate._length = lattice_length+1
#model._TranslationModel__rxn_size = 7 + 2 + 2 # manual override of rxn size

############ resources ####################
#n/a

############ states ####################
# we need two states for the mRNA FSS turning on and off:
model_pause_twostate.add_states(1, state0=[0], names=['off'])

############ Constant RXNS ####################

# ribosomal initiation
footprint = 9
# first add the reaction, in this case, we want a lattice reaction at the first
# location for a ribosome to bind (excluded)
init = lambda k,t,p,ke,o,l,pr,s,r,nr: ~np.any(l[0:0+footprint])*k[0]*(t<20000)
model_pause_twostate.add_lattice_reaction(init, parameters, rxn_name='init', exclusion=1, frame=0, loc=0, dexist=1,)

# Now we need a reaction for ribosomes to leave the lattice frame 0 at its end
leave_f0 = lambda k,t,p,ke,o,l,pr,s,r,nr: l[k[7]]*k[5]
model_pause_twostate.add_lattice_reaction(leave_f0, parameters, rxn_name='termination_frame0', exclusion=0, frame=0, loc=len(aa_seq1), dexist=-1)

# Now we need a reaction for ribosomes to leave the lattice frame 1 at its end
leave_f1 = lambda k,t,p,ke,o,l,pr,s,r,nr: l[k[8]]*k[6]
model_pause_twostate.add_lattice_reaction(leave_f1, parameters, rxn_name='termination_frame1', frame=1, loc=lattice_length, dexist=-1,)


############# state RXNS ##############
# mRNA turning on
mRNA_on = lambda k,t,p,ke,o,l,pr,s,r,nr: (1-s[0])*k[1]
model_pause_twostate.add_state_reaction(mRNA_on, parameters, rxn_name='mRNA on', inds=[0],dstates=[1])
# mRNA turning off
mRNA_off = lambda k,t,p,ke,o,l,pr,s,r,nr: s[0]*k[2]
model_pause_twostate.add_state_reaction(mRNA_off, parameters, rxn_name='mRNA off', inds=[0], dstates=[-1])


# DEFAULT STEPPING OF ELONGATION USING THE ELONGATION MATRIX                                        
default_step = lambda k,t,p,ke,o,l,pr,s,r,nr:  [(ke[p[i,2], p[i,3]])*(1 - sum(l[p[i,3]+1:p[i,3]+footprint])) for i in range(nr)]
model_pause_twostate.add_ribosome_reaction(default_step, parameters, rxn_name='default elongation', exclusion=1, dloc=1)

# FSS
# JUMP +1 frame if greater than or equal to frameshift location to fss +10, on frame 0, state = on
FSS = lambda k,t,p,ke,o,l,pr,s,r,nr: [k[10]*(p[i,2] == 0)*(p[i,3] >= k[9])*(p[i,3] <= (k[9]+10))*s[0] for i in range(nr)]
model_pause_twostate.add_ribosome_reaction(FSS, parameters, rxn_name='FSS', exclusion=1, dframe=1, dloc=0)

#initial state
#odel_pause_twostate._state_arr0[0] = 1 # FSS off
# finally specify which reactions are ribosome specific
model_pause_twostate._ribosome_reactions = [5,6]
model_pause_twostate._constant_reactions = [0,1,2,3,4]

#model_pause_twostate._lattice_arr0 = np.zeros([model._length+1], dtype=int)

try:
    model_pause_twostate.load_model('GagPol_twostate_bursting_FSS')
except:
    pass

model_pause_twostate.compile_model_c()

['c:\\Users\\willi\\anaconda3\\envs\\py312\\Library\\include\\', 'c:\\Users\\willi\\anaconda3\\envs\\py312\\Library\\include\\', 'c:\\Users\\willi\\anaconda3\\envs\\py312\\Library\\include\\', 'c:\\Users\\willi\\anaconda3\\envs\\py312\\Library\\include\\']
eigen instillation found...
['c:\\Users\\willi\\anaconda3\\envs\\py312\\Library\\include\\', 'c:\\Users\\willi\\anaconda3\\envs\\py312\\Library\\include\\', 'c:\\Users\\willi\\anaconda3\\envs\\py312\\Library\\include\\', 'c:\\Users\\willi\\anaconda3\\envs\\py312\\Library\\include\\']
['c:\\Users\\willi\\anaconda3\\envs\\py312\\Library\\include\\', 'c:\\Users\\willi\\anaconda3\\envs\\py312\\Library\\include\\', 'c:\\Users\\willi\\anaconda3\\envs\\py312\\Library\\include\\', 'c:\\Users\\willi\\anaconda3\\envs\\py312\\Library\\include\\']
converting propensities to c...
[<function <lambda> at 0x00000269C723AFC0>, <function <lambda> at 0x00000269C723A340>, <function <lambda> at 0x00000269C723A660>, <function <lambda> at 0x00000269C723B42

In [19]:
n = 10
ti, tf = 0, 40001
nt = int(tf/60)

t = np.linspace(ti,tf,nt)
soln = rsnp.solver.solve_ssa(model_pause_twostate, t, n_traj=n, burnin=0, verbose=True, cplus=True)

Using C++.....


Running mRNA simulation...: 100%|██████████| 10/10 [00:00<00:00, 87.58it/s]


In [39]:
model_pause_twostate._parameters

[[0.0244, 9.6e-05, 0.00013, 0.0234, 0.0139, 3, 3, 1021, 2032, 951, 1000000.0],
 [0.0244, 9.6e-05, 0.00013, 0.0234, 0.0139, 3, 3, 1021, 2032, 951, 1000000.0],
 [0.0244, 9.6e-05, 0.00013, 0.0234, 0.0139, 3, 3, 1021, 2032, 951, 1000000.0],
 [0.0244, 9.6e-05, 0.00013, 0.0234, 0.0139, 3, 3, 1021, 2032, 951, 1000000.0],
 [0.0244, 9.6e-05, 0.00013, 0.0234, 0.0139, 3, 3, 1021, 2032, 951, 1000000.0],
 [0.0244, 9.6e-05, 0.00013, 0.0234, 0.0139, 3, 3, 1021, 2032, 951, 1000000.0],
 [0.0244, 9.6e-05, 0.00013, 0.0234, 0.0139, 3, 3, 1021, 2032, 951, 1000000.0]]